# Notebook 5 : comparaison de schémas

**Question** : peut-on faire mieux que l'upwind ? À quel prix ?

L'upwind étale les fronts (diffusion numérique). Un schéma d'ordre supérieur les garde plus nets, mais peut créer des oscillations. On compare :

| Schéma | Idée | Temps |
|---|---|---|
| **upwind** | c_face = c de la cellule amont (constant par maille) | Euler explicite |
| **MUSCL + minmod** | on reconstruit une **pente** dans chaque cellule, limitée pour ne pas créer de nouveau max/min ; c_face est extrapolé depuis la cellule amont | RK2 (Heun) |
| **MUSCL + van Leer** | idem, avec un limiteur moins dissipatif | RK2 |
| **MUSCL + Fromm (sans limiteur)** | pente centrée non limitée : plus précis en zone lisse, mais oscille près des fronts | RK2 |

Tous restent des schémas volumes finis conservatifs : seule la façon de calculer `c_face` change. Le schéma d'ordre 2 en espace est associé à un schéma d'ordre 2 en temps (RK2 : deux évaluations du second membre par pas) pour que l'erreur en temps ne gâche pas la précision.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('figures', exist_ok=True)
from advection import *

schemas = {
    "upwind":          dict(schema='upwind', temps='euler'),
    "MUSCL minmod":    dict(schema='muscl', lim='minmod',  temps='rk2'),
    "MUSCL van Leer":  dict(schema='muscl', lim='vanleer', temps='rk2'),
    "MUSCL Fromm":     dict(schema='muscl', lim='fromm',   temps='rk2'),
}

## 1. Profil lisse (gaussienne)

In [ ]:
res_g = {nom: cas_translation(N=100, profil='gauss', **opt) for nom, opt in schemas.items()}
r = res_g["upwind"]
i_mid = r['c'].shape[0] // 2

plt.figure(figsize=(9, 4.5))
plt.plot(r['x'], r['c_exact'][i_mid], 'k-', lw=2, label="exact")
for nom, rr in res_g.items():
    plt.plot(rr['x'], rr['c'][i_mid], 'o-', ms=3, lw=1, label=nom)
plt.xlim(30, 80)
plt.xlabel("x"); plt.ylabel("c"); plt.title("Gaussienne après translation (profil au centre)")
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('figures/schemas_gauss.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"{'schéma':16s} {'max c':>8} {'min c':>11} {'erreur L2':>11}")
for nom, rr in res_g.items():
    print(f"{nom:16s} {rr['c'].max():8.4f} {rr['c'].min():11.2e} {rr['err']['L2']:11.3e}")

## 2. Profil raide (créneau) : diffusion contre oscillations

C'est ici que les différences se voient : l'upwind étale le front (diffusion numérique), les schémas limités le gardent plus net sans dépasser [0, 1], et le schéma sans limiteur **dépasse** [0, 1] (valeurs négatives ou supérieures à 1) : ce sont des oscillations de type Gibbs.

In [ ]:
res_c = {nom: cas_translation(N=200, profil='creneau', **opt) for nom, opt in schemas.items()}
r = res_c["upwind"]
i_mid = r['c'].shape[0] // 2

plt.figure(figsize=(9, 4.5))
plt.plot(r['x'], r['c_exact'][i_mid], 'k-', lw=2, label="exact")
for nom, rr in res_c.items():
    plt.plot(rr['x'], rr['c'][i_mid], 'o-', ms=3, lw=1, label=nom)
plt.xlim(35, 90)
plt.xlabel("x"); plt.ylabel("c"); plt.title("Créneau après translation (profil au centre)")
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('figures/schemas_creneau.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"{'schéma':16s} {'max c':>8} {'min c':>11} {'erreur L1':>11}")
for nom, rr in res_c.items():
    print(f"{nom:16s} {rr['c'].max():8.4f} {rr['c'].min():11.2e} {rr['err']['L1']:11.3e}")

Une valeur de c hors de [0, 1] est physiquement impossible (une concentration ne peut pas être négative). C'est le défaut d'un schéma d'ordre 2 non limité (théorème de Godunov : un schéma linéaire monotone ne peut pas dépasser l'ordre 1). Les limiteurs rendent le schéma **non linéaire** pour garder la monotonie.

## 3. Courbes de convergence superposées (gaussienne)

In [ ]:
Ns = [50, 100, 200, 400]
plt.figure(figsize=(7.5, 5.5))
print(f"{'schéma':16s} {'ordre L1':>9} {'ordre L2':>9}")
for nom, opt in schemas.items():
    d, e1, e2 = [], [], []
    for N in Ns:
        r = cas_translation(N=N, profil='gauss', **opt)
        d.append(r['dx']); e1.append(r['err']['L1']); e2.append(r['err']['L2'])
    d = np.array(d)
    print(f"{nom:16s} {ordre_observe(d, e1):9.2f} {ordre_observe(d, e2):9.2f}")
    plt.loglog(d, e2, 'o-', label=nom)
plt.loglog(d, e2[0] * (d / d[0]), 'k--', lw=0.8, label="pente 1")
plt.loglog(d, e2[0] * (d / d[0])**2, 'k:', lw=0.8, label="pente 2")
plt.xlabel("dx"); plt.ylabel("erreur L2"); plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.title("Convergence en maillage, gaussienne")
plt.savefig('figures/convergence_schemas.png', dpi=200, bbox_inches='tight')
plt.show()

Les limiteurs réduisent un peu l'ordre là où ils s'activent (aux extrema de la gaussienne) : on observe typiquement entre 1,4 et 1,7 pour minmod et van Leer, et près de 2 pour Fromm, qui n'est pas limité.

## 4. Coût contre précision

In [ ]:
N = 200
print(f"{'schéma':16s} {'temps (s)':>10} {'erreur L2':>11}")
for nom, opt in schemas.items():
    r = cas_translation(N=N, profil='gauss', **opt)
    print(f"{nom:16s} {r['duree']:10.3f} {r['err']['L2']:11.3e}")

Un schéma d'ordre 2 coûte plus cher par pas (deux évaluations du second membre avec RK2 et une reconstruction), mais il atteint une erreur donnée avec un maillage beaucoup plus grossier. Compare : à quel dx l'upwind atteint-il l'erreur obtenue par MUSCL à N = 100 ? Combien cela coûterait-il ?

## 5. Conclusion à rédiger

- Quel schéma garde le mieux le front ? Lequel dépasse [0, 1] ?
- Que gagne-t-on, et que perd-on, avec un limiteur ?
- Lequel choisirais-tu pour le labyrinthe, et pourquoi ?